# Arabic News Fine-Tuning Project

This notebook fine-tunes **Qwen2.5-1.5B-Instruct** using **LoRA** (via LLaMA-Factory)
on two tasks:
1. **Details Extraction** — read an Arabic news story and extract structured
   details (title, keywords, summary, category, entities) as JSON.
2. **Translation** — translate the story into another language, also as JSON.

The goal is to compare the base model's output on these tasks before and
after fine-tuning, using a small open-source model that can run on a free
Colab GPU.

#Setup

In [7]:
!pip install -qU transformers==4.48.3 datasets==3.2.0 optimum==1.24.0
!pip install -qU openai==1.61.0 wandb
!pip install -qU json-repair==0.29.1
!pip install -qU faker==35.2.0

In [8]:
!git clone --depth 1 https://github.com/hiyouga/LLaMA-Factory.git
!cd LLaMA-Factory && pip install -e .

fatal: destination path 'LLaMA-Factory' already exists and is not an empty directory.
Obtaining file:///content/LLaMA-Factory
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Using cached peft-0.18.1-py3-none-any.whl.metadata (14 kB)
  Using cached transformers-5.8.0-py3-none-any.whl.metadata (33 kB)
  Using cached huggingface_hub-1.32.0-py3-none-any.whl.metadata (16 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (7.3 kB)
Using cached peft-0.18.1-py3-none-any.whl (556 kB)
Using cached transformers-5.8.0-py3-none-any.whl (10.6 MB)
Using cached huggingface_hub-1.32.0-py3-none-any.whl (842 kB)
Using cached tokenizers-0.22.2-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (3.3 MB)
  Building editable for llamafactory (pyproje

In [9]:
from google.colab import drive
drive.mount('/gdrive', force_remount=True)

Mounted at /gdrive


In [10]:
!mkdir -p /gdrive/MyDrive/llm-finetuning/models
!test -f /gdrive/MyDrive/llm-finetuning/datasets/news-sample.jsonl || gdown --folder "https://drive.google.com/drive/folders/1dXNNFNg_RKMYC9nxF59d0LAt67-T4oDf" -O /gdrive/MyDrive/llm-finetuning/datasets
!ls /gdrive/MyDrive/llm-finetuning/datasets

llamafactory-finetune-data  news-sample.jsonl  sft.jsonl  xsft.jsonl


### Required Secrets
Before running the login cell, add these in Colab Secrets (🔑 icon → Notebook access ON):
- `wandb`
- `huggingface`
- `groq`

In [11]:
from google.colab import userdata
import wandb
from huggingface_hub import login

wandb.login(key=userdata.get('wandb'))
hf_token = userdata.get('huggingface')
login(token=hf_token)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: lailahamada620 (lailahamada620-student) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


# Import

In [4]:
import json
import os
from os.path import join
import random
from tqdm.auto import tqdm
import requests

from pydantic import BaseModel, Field
from typing import List, Optional, Literal
from datetime import datetime

import json_repair

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

data_dir = "/gdrive/MyDrive/llm-finetuning"
base_model_id = "Qwen/Qwen2.5-1.5B-Instruct"

device = "cuda"
torch_dtype = None

def parse_json(text):
    try:
        return json_repair.loads(text)
    except:
        return None

print(os.listdir(join(data_dir, "datasets")))
print(torch.cuda.is_available())

['news-sample.jsonl', 'sft.jsonl', 'xsft.jsonl', 'llamafactory-finetune-data']
True


###Tasks
- `structured output for Details Extraction and Translation`
- `system prompt for Details Extraction and Translation `


In [5]:
story = """
ذكرت مجلة فوربس أن العائلة تلعب دورا محوريا في تشكيل علاقة الأفراد بالمال،
 حيث تتأثر هذه العلاقة بأنماط السلوك المالي المتوارثة عبر الأجيال.

التقرير الذي يستند إلى أبحاث الأستاذ الجامعي شاين إنيت حول
الرفاه المالي يوضح أن لكل شخص "شخصية مالية" تتحدد وفقا لطريقة
 تفاعله مع المال، والتي تتأثر بشكل مباشر بتربية الأسرة وتجارب الطفولة.

 الأبعاد الثلاثة للعلاقة بالمال
بحسب الدراسة، هناك ثلاثة أبعاد رئيسية تشكّل علاقتنا بالمال:

الاكتساب (A): يميل الأفراد الذين ينتمون لهذا
 البعد إلى اعتبار المال سلعة قابلة للجمع، حيث يرون
في تحقيق الثروة هدفا بحد ذاته. والجانب السلبي لهذا
 النمط هو إمكانية التحول إلى هوس بالثروة أو العكس،
 أي رفض تام لاكتساب المال باعتباره مصدرا للفساد.

الاستخدام (U): يرى هؤلاء الأشخاص المال أداة للتمتع بالحياة، حيث يربطون قيمته بقدرته على توفير
المتعة والراحة. ومع ذلك، قد يصبح
البعض مدمنا على الإنفاق، في حين يتجه آخرون إلى التقشف المفرط خوفا من المستقبل.

الإدارة (M): أصحاب هذا النمط يعتبرون المال مسؤولية تتطلب التخطيط الدقيق. لكن في بعض الحالات،
 قد يتحول الأمر إلى هوس مفرط بإدارة الإنفاق، مما يؤثر سلبا على العلاقات الشخصية.

 كيف تؤثر العائلة على علاقتنا بالمال؟
يشير التقرير إلى أن التجارب الأسرية تلعب دورا رئيسيا في تحديد
 "الشخصية المالية" لكل فرد، على سبيل المثال، إذا كان أحد الوالدين يعتمد على المال
كمكافأة للسلوك الجيد، فقد يتبنى الطفل لاحقا النمط نفسه في حياته البالغة.

لتحليل هذه التأثيرات بشكل دقيق، طورت رابطة العلاج المالي
(Financial Therapy Association) أداة تسمى مخطط الجينوم المالي (Money Genogram)،
وهو نموذج يُستخدم لتحديد الأنماط المالية داخل العائلة.

تتضمن هذه الأداة:

رسم شجرة عائلية.
تصنيف أفراد العائلة وفقا للأبعاد الثلاثة للعلاقة بالمال (A ،U ،M).
تحديد ما إذا كان السلوك المالي لكل فرد صحيا (+) أو غير صحي (-).
على سبيل المثال، إذا نشأ شخص في عائلة
اعتادت على الإنفاق المفرط، فقد يكون لديه ميل قوي إلى اتباع النمط نفسه،
 أو العكس تماما، حيث يصبح مقتصدا بشكل مبالغ فيه كرد فعل نفسي.
"""

#Details Extraction


In [6]:
StoryCategory = Literal[
    "politics", "sports", "art", "technology", "economy",
    "health", "entertainment", "science",
    "not_specified"
]

EntityType = Literal[
    "person-male", "person-female", "location", "organization", "event", "time",
    "quantity", "money", "product", "law", "disease", "artifact", "not_specified"
]

class Entity(BaseModel):
    entity_value: str = Field(..., description="The actual name or value of the entity.")
    entity_type: EntityType = Field(..., description="The type of recognized entity.")

class NewsDetails(BaseModel):
    story_title: str = Field(..., min_length=5, max_length=300,
                             description="A fully informative and SEO optimized title of the story.")

    story_keywords: List[str] = Field(..., min_items=1,
                                      description="Relevant keywords associated with the story.")

    story_summary: List[str] = Field(
                                    ..., min_items=1, max_items=5,
                                    description="Summarized key points about the story (1-5 points)."
                                )

    story_category: StoryCategory = Field(..., description="Category of the news story.")

    story_entities: List[Entity] = Field(..., min_items=1, max_items=10,
                                        description="List of identified entities in the story.")


/tmp/ipykernel_64785/2691156874.py:20: PydanticDeprecatedSince20: `min_items` is deprecated and will be removed, use `min_length` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  story_keywords: List[str] = Field(..., min_items=1,
/tmp/ipykernel_64785/2691156874.py:23: PydanticDeprecatedSince20: `min_items` is deprecated and will be removed, use `min_length` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  story_summary: List[str] = Field(
/tmp/ipykernel_64785/2691156874.py:23: PydanticDeprecatedSince20: `max_items` is deprecated and will be removed, use `max_length` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  story_summary: List[str] = Field(
/tmp/ipykernel_64785/2691156874.py:30: PydanticDeprecatedSince20: `

In [7]:
details_extraction_messages = [
    {
        "role": "system",
        "content": "\n".join([
            "You are an NLP data paraser.",
            "You will be provided by an Arabic text associated with a Pydantic scheme.",
            "Generate the ouptut in the same story language.",
            "You have to extract JSON details from text according the Pydantic details.",
            "Extract details as mentioned in text.",
            "Do not generate any introduction or conclusion."
        ])
    },
    {
        "role": "user",
        "content": "\n".join([
            "## Story:",
            story.strip(),
            "",

            "## Pydantic Details:",
            json.dumps(
                NewsDetails.model_json_schema(), ensure_ascii=False
            ),
            "",

            "## Story Details:",
            "```json"
        ])
    }
]

#Translation

In [8]:
class TranslatedStory(BaseModel):
    translated_title: str = Field(..., min_length=5, max_length=300,
                                  description="Suggested translated title of the news story.")
    translated_content: str = Field(..., min_length=5,
                                    description="Translated content of the news story.")

targeted_lang = "English"

translation_messages = [
    {
        "role": "system",
        "content": "\n".join([
            "You are a professional translator.",
            "You will be provided by an Arabic text.",
            "You have to translate the text into the `Targeted Language`.",
            "Follow the provided Scheme to generate a JSON",
            "Do not generate any introduction or conclusion."
        ])
    },
    {
        "role": "user",
        "content":  "\n".join([
            "## Story:",
            story.strip(),
            "",

            "## Pydantic Details:",
            json.dumps( TranslatedStory.model_json_schema(), ensure_ascii=False ),
            "",

            "## Targeted Language:",
            targeted_lang,
            "",

            "## Translated Story:",
            "```json"

        ])
    }
]

## Evaluate Qwen2.5-1.5B-Instruct

In [17]:
model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    device_map="auto",
    torch_dtype = torch_dtype
)

tokenizer = AutoTokenizer.from_pretrained(base_model_id)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [18]:
text = tokenizer.apply_chat_template(
    details_extraction_messages,
    tokenize=False,
    add_generation_prompt=True
)

model_inputs = tokenizer([text], return_tensors="pt").to(device)

generated_ids = model.generate(
    model_inputs.input_ids,
    max_new_tokens=1024,
    do_sample=False, top_k=None, temperature=None, top_p=None,
)

generated_ids = [
    output_ids[len(input_ids):]
    for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]

response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


In [19]:
text = tokenizer.apply_chat_template(
    translation_messages,
    tokenize=False,
    add_generation_prompt=True
)

model_inputs = tokenizer([text], return_tensors="pt").to(device)

generated_ids = model.generate(
    model_inputs.input_ids,
    max_new_tokens=1024,
    do_sample=False, top_k=None, temperature=None, top_p=None,
)

generated_ids = [
    output_ids[len(input_ids):]
    for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]

response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

## Evaluate OpenAI

In [20]:
from openai import OpenAI
from google.colab import userdata

openai_client = OpenAI(
    api_key=userdata.get('groq'),
    base_url="https://api.groq.com/openai/v1"
)

openai_model_id = "openai/gpt-oss-120b"

In [21]:
chat_completion = openai_client.chat.completions.create(
    messages=details_extraction_messages,
    model=openai_model_id,
    temperature=0.2,
)

print(chat_completion.choices[0].message.content)

{
  "story_title": "دور العائلة في تشكيل الشخصية المالية وأبعاد العلاقة بالمال وفقًا لتقرير فوربس",
  "story_keywords": [
    "العائلة",
    "الشخصية المالية",
    "العلاقة بالمال",
    "مخطط الجينوم المالي",
    "فوربس",
    "الرفاه المالي",
    "الأبعاد الثلاثة",
    "الاكتساب",
    "الاستخدام",
    "الإدارة"
  ],
  "story_summary": [
    "تؤثر العائلة بشكل محوري في تكوين \"الشخصية المالية\" للفرد عبر أنماط سلوك مالي موروثة.",
    "تحدد الدراسة ثلاثة أبعاد رئيسية للعلاقة بالمال: الاكتساب (A)، الاستخدام (U)، والإدارة (M).",
    "كل بعد يحمل جوانب إيجابية وسلبية قد تؤدي إلى هوس أو رفض أو تقشف مفرط.",
    "طورت رابطة العلاج المالي أداة \"مخطط الجينوم المالي\" لتصنيف سلوكيات أفراد العائلة وفق الأبعاد الثلاثة وتقييم صحتها.",
    "التجارب الأسرية مثل مكافأة السلوك الجيد بالمال تشكل نمطًا ماليًا مستقبليًا للطفل."
  ],
  "story_category": "economy",
  "story_entities": [
    {
      "entity_value": "مجلة فوربس",
      "entity_type": "organization"
    },
    {
      "entity_value": "شاين إني

In [22]:
chat_completion = openai_client.chat.completions.create(
    messages=translation_messages,
    model=openai_model_id,
    temperature=0.2,
)

print(chat_completion.choices[0].message.content)

```json
{
  "translated_title": "The Role of Family in Shaping Individuals' Relationship with Money",
  "translated_content": "Forbes magazine reported that the family plays a pivotal role in shaping individuals' relationship with money, as this relationship is influenced by financial behavior patterns inherited across generations.\n\nThe report, based on research by university professor Shane Enit on financial well‑being, explains that each person has a \"financial personality\" determined by how they interact with money, which is directly affected by family upbringing and childhood experiences.\n\n**The Three Dimensions of the Money Relationship**\nAccording to the study, there are three main dimensions that shape our relationship with money:\n\n*Acquisition (A):* Individuals who fall into this dimension tend to view money as a collectible commodity, seeing the accumulation of wealth as an end in itself. The downside of this pattern is the potential to become obsessed with wealth—or,

## Knowledge Distillation

Fine-tuning needs thousands of labeled examples (story → correct JSON output),
which is too many to write by hand. So instead, a larger cloud model (the
"teacher") is given each raw news story and asked to produce the ideal output
for both tasks. Its answers are saved to `sft.jsonl` and later used to train
the small local model (the "student") to imitate this behavior.

**Note:** `sft.jsonl` is already generated for this project (loaded from
Drive). This section is kept for reference only — do not re-run it, as it
would duplicate entries and re-spend API credits.

In [23]:
raw_data_path = join(data_dir, "datasets", "news-sample.jsonl")

raw_data = []
for line in open(raw_data_path):
    if line.strip() == "":
        continue

    raw_data.append(
        json.loads(line.strip())
    )

random.Random(101).shuffle(raw_data)

print(f"Raw data: {len(raw_data)}")

Raw data: 2400


In [24]:
raw_data[0]['content']

'يواصل المعهد العربي في باريس استقبال زواره في معرض ما تقدمه فلسطين للعالم لإطلاعهم على الإرث الثقافي والفني للفلسطينيين؛ من خلال أعمال فنية لآمالهم وصور لواقعهم الأليم تحت الاحتلال. \n ويرى رئيس المعهد جاك لانغ -الذي أُعيد انتخابه قبل أيام للدورة الرابعة- ما يحدث في غزة حاليا جراء العدوان الإسرائيلي أنه كارثة. \n والمعهد هو مركز ثقافي وواجهة دبلوماسية يديرها لانغ منذ 2013 ويقع على ضفة نهر السين في باريس. \n وأشار لانغ، الذي شغل سابقا منصب وزير الثقافة بفرنسا، إلى أن المعرض هو إهداء للشعب الفلسطيني، ومُدّد ليستقبل مزيدا من الزوار حتى 31 ديسمبركانون الأول الجاري. \n ويضم المعرض، الذي افتُتح أواخر مايوأيار الماضي، حسب لانغ العديد من المعارض الفرعية عن فلسطين وعن غزة بالتحديد، من بينها معرض الصور اليومية عن الحياة في غزة. \n كما يشتمل على معرض الصور الفوتوكرومية القائم على تلوين صور من فلسطين تعود للقرن الـ19. \n ويعرض الفنان الفلسطيني محمد أبو سل عملا فريدا بعنوان مترو غزة، وهو عبارة عن عمل تركيبي متعدد الوسائط، لاقى إعجابا من الزوار. \n ويحضر الشاعر الفلسطيني الراحل محمود درويش من خلال 

In [25]:
sft_path = join(data_dir, "datasets", "sft.jsonl")
print(sum(1 for _ in open(sft_path)))

2766


## Format Finetuning Datasets

In [26]:

sft_data_path = join(data_dir, "datasets", "sft.jsonl")
llm_finetunning_data = []

system_message = "\n".join([
    "You are a professional NLP data parser.",
    "Follow the provided `Task` by the user and the `Output Scheme` to generate the `Output JSON`.",
    "Do not generate any introduction or conclusion."
])

for line in open(sft_data_path):
    if line.strip() == "":
        continue

    rec = json.loads(line.strip())

    llm_finetunning_data.append({
        "system": system_message,
        "instruction": "\n".join([
            "# Story:",
            rec["story"],

            "# Task:",
            rec["task"],

            "# Output Scheme:",
            rec["output_scheme"],
            "",

            "# Output JSON:",
            "```json"

        ]),
        "input": "",
        "output": "\n".join([
            "```json",
            json.dumps(rec["response"], ensure_ascii=False, default=str),
            "```"
        ]),
        "history": []
    })

random.Random(101).shuffle(llm_finetunning_data)

In [27]:
len(llm_finetunning_data)

2766

In [28]:
train_sample_sz = 2700

train_ds = llm_finetunning_data[:train_sample_sz]
eval_ds = llm_finetunning_data[train_sample_sz:]

os.makedirs(join(data_dir, "datasets", "llamafactory-finetune-data"), exist_ok=True)

with open(join(data_dir, "datasets", "llamafactory-finetune-data", "train.json"), "w") as dest:
    json.dump(train_ds, dest, ensure_ascii=False, default=str)

with open(join(data_dir, "datasets", "llamafactory-finetune-data", "val.json"), "w", encoding="utf8") as dest:
    json.dump(eval_ds, dest, ensure_ascii=False, default=str)

In [29]:
join(data_dir, "datasets", "llamafactory-finetune-data", "val.json")

'/gdrive/MyDrive/llm-finetuning/datasets/llamafactory-finetune-data/val.json'

In [30]:
%%writefile /content/LLaMA-Factory/examples/train_lora/news_finetune.yaml

### model
model_name_or_path: Qwen/Qwen2.5-1.5B-Instruct
trust_remote_code: true

### method
stage: sft
do_train: true
finetuning_type: lora
lora_rank: 16
lora_target: all

### dataset
dataset: news_finetune_train
eval_dataset: news_finetune_val
template: qwen
cutoff_len: 3500
# max_samples: 50
overwrite_cache: true
preprocessing_num_workers: 16

### output
# resume_from_checkpoint: /gdrive/MyDrive/llm-finetuning/models/checkpoint-1500
output_dir: /gdrive/MyDrive/llm-finetuning/models/
logging_steps: 10
save_steps: 500
plot_loss: true
# overwrite_output_dir: true

### train
per_device_train_batch_size: 1
gradient_accumulation_steps: 4
learning_rate: 1.0e-4
num_train_epochs: 3.0
lr_scheduler_type: cosine
warmup_ratio: 0.1
bf16: true
ddp_timeout: 180000000

### eval
# val_size: 0.1
per_device_eval_batch_size: 1
eval_strategy: steps
eval_steps: 100

report_to: wandb
run_name: newsx-finetune-llamafactory

push_to_hub: false
export_hub_model_id: "laila66/news-analyzer"
hub_private_repo: true
hub_strategy: checkpoint


Overwriting /content/LLaMA-Factory/examples/train_lora/news_finetune.yaml


In [ ]:
!cd LLaMA-Factory/ && llamafactory-cli train /content/LLaMA-Factory/examples/train_lora/news_finetune.yaml

## New Finetuned Model Evaluation

In [ ]:
!gdown --folder "https://drive.google.com/drive/folders/1IIEB8BaQpMf8P5BQCmMGpdmOQst7OxHf" -O /gdrive/MyDrive/llm-finetuning/models
!ls /gdrive/MyDrive/llm-finetuning/models

In [33]:
from google.colab import drive
drive.mount('/gdrive', force_remount=True)

Mounted at /gdrive


In [34]:
from google.colab import userdata
import wandb
from huggingface_hub import login

wandb.login(key=userdata.get('wandb'))
hf_token = userdata.get('huggingface')
login(token=hf_token)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


In [35]:
model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    device_map="auto",
    torch_dtype = torch_dtype
)

tokenizer = AutoTokenizer.from_pretrained(base_model_id)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [37]:
!pip install -qU torchao

In [ ]:
!pip install -U transformers peft

In [39]:
import transformers
import peft

print("transformers:", transformers.__version__)
print("peft:", peft.__version__)

transformers: 5.8.0
peft: 0.21.0


In [40]:
finetuned_model_id = "/gdrive/MyDrive/llm-finetuning/models"
model.load_adapter(finetuned_model_id)

Loading weights:   0%|          | 0/392 [00:00<?, ?it/s]

LoadStateDictInfo(missing_keys=set(), unexpected_keys=set(), mismatched_keys=set(), error_msgs=[], conversion_errors={})

In [41]:
def generate_resp(messages):
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    model_inputs = tokenizer([text], return_tensors="pt").to(device)

    generated_ids = model.generate(
        model_inputs.input_ids,
        max_new_tokens=1024,
        do_sample=False, top_k=None, temperature=None, top_p=None,
    )

    generated_ids = [
        output_ids[len(input_ids):]
        for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]

    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

    return response

response = generate_resp(translation_messages)

In [42]:
parse_json(response)

{'translated_title': 'The Role of Family in Financial Relationships',
 'translated_content': 'Forbes magazine reported that the family plays a pivotal role in shaping individuals\' relationship with money, as this relationship is influenced by inherited financial behaviors across generations.\n\nThe report, based on research by Professor Shane Everette on financial well-being, explains that each person has a \'financial personality\' determined by how they interact with money, which is directly affected by family upbringing and childhood experiences.\n\nThe three dimensions of the money relationship\nAccording to the study, there are three main dimensions that form our relationship with money:\n\nAcquisition (A): Individuals belonging to this dimension tend to view money as a commodity that can be accumulated, seeing wealth accumulation as a goal in itself. The downside of this pattern is the potential for it to turn into an obsession with wealth or vice versa, meaning complete rejecti

###Cost Estimation

In [43]:
from tqdm.auto import tqdm
from faker import Faker
import random
from datetime import datetime

start_time = datetime.now()
fake = Faker('ar')

input_tokens = 0
output_tokens = 0

for i in tqdm(range(30)):
    prompt = fake.text(max_nb_chars=random.randint(150, 200))

    messages = [
        {
            "role": "user",
            "content": prompt,
        }
    ]

    response = generate_resp(messages)

    input_tokens += len(tokenizer.apply_chat_template(messages))
    output_tokens += len(tokenizer.encode(response))

total_time = (datetime.now() - start_time).total_seconds()

print(f"Total Time: {total_time} seconds")
print(f"Input Tokens: {input_tokens}")
print(f"Output Tokens: {output_tokens}")
print(f"Total Tokens: {input_tokens + output_tokens}")

  0%|          | 0/30 [00:00<?, ?it/s]

Total Time: 827.434704 seconds
Input Tokens: 60
Output Tokens: 14014
Total Tokens: 14074


## *vLLM*

In [1]:
import transformers
import vllm
import importlib.metadata as metadata

print("Transformers:", transformers.__version__)
print("Transformers metadata:", metadata.version("transformers"))
print("vLLM:", vllm.__version__)

Transformers: 5.17.0
Transformers metadata: 5.17.0
vLLM: 0.30.0


In [2]:
!nohup vllm serve "Qwen/Qwen2.5-1.5B-Instruct" \
  --dtype=half \
  --gpu-memory-utilization=0.8 \
  --max-lora-rank=64 \
  --enable-lora \
  --lora-modules news-lora="/gdrive/MyDrive/llm-finetuning/models" \
  > nohup.out 2>&1 &

In [11]:
!tail -n 40 nohup.out

(EngineCore pid=64590) INFO 09-22 21:21:39 [gpu_worker.py:825] CUDA graph pool memory: 0.33 GiB (actual), 0.71 GiB (estimated), difference: 0.38 GiB (115.0%).
(EngineCore pid=64590) INFO 09-22 21:21:39 [gpu_worker.py:888] Free memory on device (14.46/14.56 GiB) on startup. Desired GPU memory utilization is (0.8, 11.65 GiB). Actual usage is 3.89 GiB for consumed memory (weights + non-torch), 1.15 GiB for peak activation, and 0.33 GiB for CUDAGraph memory. Replace gpu_memory_utilization config with `--kv-cache-memory=6584423527` (6.13 GiB) to fit into requested memory, or `--kv-cache-memory=9601871360` (8.94 GiB) to fully utilize gpu memory. Current kv cache memory in use is 6.61 GiB.
(EngineCore pid=64590) INFO 09-22 21:21:40 [jit_monitor.py:85] Kernel JIT monitor activated; monitored JIT compilations during inference will use mode=warn.
(EngineCore pid=64590) INFO 09-22 21:21:41 [core.py:372] init engine (profile, create kv cache, warmup model) took 289.08 s (compilation: 95.92 s)
(Eng

### Inference


In [9]:
tokenizer = AutoTokenizer.from_pretrained(base_model_id)

prompt = tokenizer.apply_chat_template(
    translation_messages,
    tokenize=False,
    add_generation_prompt=True
)

In [12]:
vllm_model_id = "news-lora"

llm_response = requests.post("http://localhost:8000/v1/completions", json={
    "model": vllm_model_id,
    "prompt": prompt,
    "max_tokens": 1000,
    "temperature": 0.3
})

llm_response.json()

{'id': 'cmpl-91f7eb56a7e0616c',
 'object': 'text_completion',
 'created': 1790112127,
 'model': 'news-lora',
 'choices': [{'index': 0,
   'text': '```json{"translated_title": "The Role of Family in Financial Relationships", "translated_content": "Forbes magazine reported that the family plays a crucial role in shaping individuals\' relationship with money, as this relationship is influenced by inherited financial behaviors across generations.\\n\\nThe report, based on research by Professor Shane Everette on financial well-being, explains that each person has a \'financial personality\' determined by how they interact with money, which is directly affected by family upbringing and childhood experiences.\\n\\nThe three dimensions of our financial relationship according to the study include:\\n\\nAcquisition (A): Individuals belonging to this dimension tend to view money as a commodity that can be accumulated, seeing wealth accumulation as a goal in itself. The downside of this pattern is

## Load Testing

In [16]:
!pip install -q locust

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.3/47.3 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 36.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 44.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.4/115.4 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.5/82.5 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 270.6/270.6 kB 13.5 MB/s eta 0:00:00


In [17]:
%%writefile locust.py

import random
import json
from locust import HttpUser, task, between, constant
from transformers import AutoTokenizer
from faker import Faker

fake = Faker('ar')

class CompletionLoadTest(HttpUser):
    wait_time = between(1, 3)

    @task
    def post_completion(self):
        model_id = "news-lora"
        prompt = fake.text(max_nb_chars=random.randint(150, 200))

        message = {
            "model": model_id,
            "prompt": prompt,
            "max_tokens": 512,
            "temperature": 0.3
        }

        llm_response = self.client.post("/v1/completions", json=message)

        if llm_response.status_code == 200:
            with open("./vllm_tokens.txt", "a") as dest:
                dest.write(json.dumps({
                    "prompt": prompt,
                    "response": llm_response.json()["choices"][0]["text"],
                }, ensure_ascii=False) + "\n")


Overwriting locust.py


In [18]:
!locust --headless -f locust.py --host=http://localhost:8000 -u 20 -r 1 -t "60s" --html=locust_results.html

[2026-09-22 21:25:46,197] abf55928eb21/INFO/locust.main: Starting Locust 2.46.6
[2026-09-22 21:25:46,198] abf55928eb21/INFO/locust.main: Run time limit set to 60 seconds
Type     Name  # reqs      # fails |    Avg     Min     Max    Med |   req/s  failures/s
--------||-------|-------------|-------|-------|-------|-------|--------|-----------
--------||-------|-------------|-------|-------|-------|-------|--------|-----------
         Aggregated       0     0(0.00%) |      0       0       0      0 |    0.00        0.00

[2026-09-22 21:25:46,202] abf55928eb21/INFO/locust.runners: Ramping to 20 users at a rate of 1.00 per second
Type     Name  # reqs      # fails |    Avg     Min     Max    Med |   req/s  failures/s
--------||-------|-------------|-------|-------|-------|-------|--------|-----------
--------||-------|-------------|-------|-------|-------|-------|--------|-----------
         Aggregated       0     0(0.00%) |      0       0       0      0 |    0.00        0.00

Type     Na

In [19]:
vllm_tokens = [
    json.loads(line.strip())
    for line in open("./vllm_tokens.txt") if line.strip() != ""
]

In [20]:
base_model_id = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(base_model_id)

total_input_tokens = sum([ len(tokenizer.encode(rec['prompt'])) for rec in vllm_tokens ])
total_output_tokens = sum([ len(tokenizer.encode(rec['response'])) for rec in vllm_tokens ])

print(f"Total Input Tokens: {total_input_tokens}")
print(f"Total Output Tokens: {total_output_tokens}")

Total Input Tokens: 167
Total Output Tokens: 914
